<a href="https://colab.research.google.com/github/rantawadeesritakorn-tech/Project-Hotel/blob/%E0%B8%84%E0%B8%99%E0%B8%97%E0%B8%B5%E0%B9%88-3/%E0%B8%84%E0%B8%99%E0%B8%97%E0%B8%B5%E0%B9%88_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ส่วนที่ 5 — class Hotel และการจำลองข้อมูลทีละรายการ

loop จำลองเดินตามปฏิทินทีละวัน ในแต่ละวันมีลูกค้าติดต่อเข้ามาจำนวนหนึ่ง
แล้วระบบรับจองทีละรายตามลำดับเวลา



In [ ]:
class Hotel:
    """ตัวโรงแรมเอง - ดูแลห้อง ลูกค้า และใบจองทั้งหมด"""

    def __init__(self, name):
        self.name = name
        self.rooms = []
        self.guests = {}                                   # dict: guest_id -> Guest
        self.bookings = []                                 # list ของ Booking ที่สำเร็จ
        self.rejected_log = []                              # เก็บรายการที่จองไม่ได้เพราะห้องเต็ม

    def add_room(self, room):
        self.rooms.append(room)

    def add_guest(self, guest):
        self.guests[guest.guest_id] = guest

    def find_available_room(self, room_type, check_in, nights):
        """หาห้องว่างประเภทที่ต้องการ - ถ้าไม่มีเลย return None"""
        candidates = [
            r for r in self.rooms
            if r.room_type == room_type
            and r.is_available(check_in, nights)
        ]

        if not candidates:
            return None
        # นโยบายโรงแรม: กระจายการใช้ห้องแบบสุ่ม ไม่ให้ห้องใดห้องหนึ่งสึกหรอเร็วเกินไป
        return random.choice(candidates)

    def occupancy_on(self, day):
        """อัตราการเข้าพักของ 'วันนั้น' -> ใช้ตัดสินใจปรับราคาแบบ dynamic pricing"""
        occupied = sum(
            1 for r in self.rooms
            if day in r.booked_dates
        )
        return occupied / len(self.rooms)

    def create_booking(
        self,
        guest,
        room_type,
        check_in,
        nights,
        channel,
        booking_date=None,
        breakfast=False,
        adults=2,
        children=0,
        rate_multiplier=1.0
    ):
        """พยายามสร้างใบจอง 1 ใบ:
        - ถ้าห้องประเภทที่ขอเต็ม -> ลองอัปเกรดให้ฟรี (เงื่อนไขพิเศษ)
        - ถ้ายังไม่ได้อีก -> ปฏิเสธ แล้วบันทึกลง rejected_log (error handling)
        """
        upgraded = False
        room = self.find_available_room(
            room_type,
            check_in,
            nights
        )

        if room is None and room_type in UPGRADE_PATH:
            room = self.find_available_room(
                UPGRADE_PATH[room_type],
                check_in,
                nights
            )
            if room is not None:
                upgraded = True

        if room is None:
            self.rejected_log.append({
                "guest_id": guest.guest_id,
                "room_type": room_type,
                "check_in": check_in.isoformat(),
                "nights": nights,
                "reason": "ห้องเต็มทุกประเภทที่รองรับได้"
            })
            return None

        booking = Booking(
            len(self.bookings) + 1,
            guest,
            room,
            check_in,
            nights,
            channel,
            booking_date or check_in,
            breakfast,
            upgraded,
            adults,
            children,
            rate_multiplier
        )

        room.reserve(check_in, nights)        # ยึดวันในปฏิทินของห้องนั้น
        guest.record_stay()                   # ลูกค้าได้แต้ม / เลื่อนระดับสมาชิก
        self.bookings.append(booking)

        return booking

    def occupancy_rate(self, start, end):
        """อัตราการเข้าพัก = คืนที่ถูกใช้จริง / คืนทั้งหมดที่ขายได้"""
        total_nights = len(self.rooms) * (
            (end - start).days + 1
        )

        sold_nights = sum(
            1
            for r in self.rooms
            for d in r.booked_dates
            if start <= d <= end
        )

        return round(
            sold_nights / total_nights * 100,
            2
        )

    def adr(self, start, end):
        """ADR (Average Daily Rate) = รายได้ค่าห้อง / จำนวนคืนที่ขายได้
        เป็นตัวชี้วัดมาตรฐานของอุตสาหกรรมโรงแรม ใช้เทียบกับตลาดจริงได้"""
        revenue = 0.0
        nights = 0

        for b in self.bookings:
            if b.is_revenue() and start <= b.check_in <= end:
                revenue += b.price_detail["room_charge"]
                nights += b.nights

        return round(
            revenue / nights,
            2
        ) if nights else 0.0

    def revpar(self, start, end):
        """RevPAR = ADR x อัตราการเข้าพัก (รายได้ต่อห้องที่มีอยู่)"""
        return round(
            self.adr(start, end)
            * self.occupancy_rate(start, end)
            / 100,
            2
        )

    def __repr__(self):
        return (
            f"Hotel({self.name}, "
            f"{len(self.rooms)} ห้อง, "
            f"{len(self.bookings)} ใบจอง)"
        )


##LOOP จำลองทีละรายการ

In [ ]:
import random
from datetime import timedelta

def create_walk_in_guest(guest_id):
    """
    สร้างลูกค้าใหม่ 1 คน โดยสุ่มสัญชาติตามสัดส่วนตลาดจริงก่อน แล้วค่อยสุ่มชื่อให้ตรงชาติ

    ยังไม่บันทึกเข้าระบบทันที - จะบันทึกก็ต่อเมื่อจองสำเร็จจริง
    (คนที่โทรมาถามแล้วไม่จอง โรงแรมไม่เก็บเป็นลูกค้าในระบบ)
    """
    nat = pick_nationality()
    return Guest(
        guest_id,
        generate_guest_name(nat),
        generate_phone(nat),
        nat
    )

def run_simulation(
    hotel,
    target_bookings=None,
    repeat_rate=0.22,
    verbose=False
):
    """
    จำลองการรับจองทีละรายการตามลำดับเวลา

    target_bookings : ถ้าระบุ จะหยุดเมื่อได้ครบจำนวนนั้น (ปกติปล่อยให้เดินจนจบช่วงเวลา)
    repeat_rate     : สัดส่วนลูกค้าเก่าที่กลับมาพักซ้ำ (โรงแรมจริงราว 20-30%)
    คืนค่า dict สรุปสถิติของรอบจำลอง
    """
    next_guest_id = 1
    requests = 0

    # เปิดรับจองล่วงหน้าตั้งแต่ก่อนช่วงที่จำลองจริง เพื่อให้มีใบจองที่ lead time ยาว
    booking_start = (
        SIM_START - timedelta(days=BOOKING_WINDOW_DAYS)
    )
    # ----- เดินเวลาทีละวัน: 1 รอบของ loop นอก = 1 วันปฏิทิน -----
    for today in daterange(
        booking_start,
        SIM_END
    ):
        # จำนวนคำขอจองที่เข้ามาในวันนี้ (สุ่มแบบ Poisson ไม่เท่ากันทุกวัน)
        n_requests = poisson(
            DAILY_REQUEST_RATE
        )

        for _ in range(n_requests):
            # ----- 1 รอบของ loop ใน = ลูกค้า 1 รายติดต่อเข้ามา -----
            if (
                target_bookings
                and len(hotel.bookings) >= target_bookings
            ):
                return summarize(
                    hotel,
                    requests
                )

            requests += 1

            # 1) ลูกค้าเป็นใคร: ลูกค้าเก่ากลับมาซ้ำ หรือลูกค้าใหม่
            is_returning = (
                bool(hotel.guests)
                and random.random() < repeat_rate
            )

            if is_returning:
                guest = random.choice(
                    list(hotel.guests.values())
                )
            else:
                guest = create_walk_in_guest(
                    next_guest_id
                )

            nat = guest.nationality

            # 2) ลูกค้ารายนี้อยากได้อะไร (พฤติกรรมขึ้นกับสัญชาติและกลุ่มผู้เดินทาง)
            # โรงแรมจริงมีลูกค้า walk-in เดินเข้ามาขอห้องวันนั้นเลยอยู่ประมาณ 7%
            lead = (
                0
                if random.random() < 0.07
                else random_lead_time(nat)
            )

            check_in = (
                today + timedelta(days=lead)
            )

            if (
                check_in > SIM_END
                or check_in < SIM_START
            ):
                continue            # วันเข้าพักอยู่นอกช่วงที่เราจำลอง (ก่อน 1 ม.ค. หรือหลัง 30 มิ.ย.)

            # ถ่วงน้ำหนักตามดีมานด์จริงของวันนั้น (ศุกร์-เสาร์และ high season คนเยอะกว่า)
            if random.random() > min(
                demand_factor(check_in) / 1.9,
                1.0
            ):
                continue           # วันนั้นดีมานด์ต่ำ ลูกค้ารายนี้ไม่จอง

            party_kind, adults, children = (
                pick_travel_party()
            )

            room_type = pick_room_type(
                party_kind
            )

            nights = random_nights(nat)

            channel = pick_channel(
                nat,
                lead
            )

            breakfast = wants_breakfast(
                nat
            )
            # 3) โรงแรมตั้งราคาตามดีมานด์ ณ ตอนนั้น (revenue management)
            occ = hotel.occupancy_on(
                check_in
            )

            rate_mult = dynamic_rate_multiplier(
                occ
            )
            # 4) ส่งให้ระบบตัดสินใจว่ารับจองได้ไหม
            booking = hotel.create_booking(
                guest,
                room_type,
                check_in,
                nights,
                channel,
                booking_date=today,
                breakfast=breakfast,
                adults=adults,
                children=children,
                rate_multiplier=rate_mult
            )

            if booking is None:
                continue                               # ห้องเต็ม -> บันทึกไว้แล้วใน rejected_log

            if not is_returning:                   # จองสำเร็จแล้วค่อยขึ้นทะเบียนลูกค้าใหม่
                hotel.add_guest(guest)
                next_guest_id += 1

            if verbose:
                print(booking)

            # 5) หลังจอง: ลูกค้าอาจยกเลิกทีหลัง (อัตราขึ้นกับช่องทางที่จองมา)
            cancel_prob = (
                CANCEL_RATE_BY_CHANNEL[channel]
            )

            if lead > 30:
                cancel_prob *= 1.25            # จองล่วงหน้านาน ยิ่งมีโอกาสเปลี่ยนใจ
            elif lead <= 3:
                cancel_prob *= 0.4             # จองกระชั้น แทบไม่ยกเลิก

            if random.random() < cancel_prob:
                # ยกเลิกก่อนถึงวันเข้าพัก (เฉลี่ยแจ้งล่วงหน้าพอสมควร)
                days_before = random.randint(
                    0,
                    max(lead, 1)
                )

                booking.cancel(
                    check_in
                    - timedelta(days=days_before)
                )
            elif random.random() < NO_SHOW_RATE:
                booking.mark_no_show()                  # ไม่มาเช็คอินโดยไม่แจ้ง
            elif booking.check_out <= SIM_END:
                booking.check_out_guest()               # พักจบแล้วก่อนวันสุดท้ายที่จำลอง

    return summarize(
        hotel,
        requests
    )


def summarize(hotel, requests):
    """สรุปสถิติของรอบจำลอง -> คืนเป็น dict"""
    counts = {}

    for b in hotel.bookings:
        counts[b.status] = (
            counts.get(b.status, 0) + 1
        )

    return {
        "requests": requests,
        "bookings": len(hotel.bookings),
        "rejected": len(hotel.rejected_log),
        "guests": len(hotel.guests),
        "status_counts": counts,
        "occupancy": hotel.occupancy_rate(
            SIM_START,
            SIM_END
        ),
        "adr": hotel.adr(
            SIM_START,
            SIM_END
        ),
        "revpar": hotel.revpar(
            SIM_START,
            SIM_END
        ),
    }


## รันการจำลองระบบและเเสดงผล

In [ ]:
random.seed(RANDOM_SEED)

hotel = Hotel("Bangkok Hotel")

for r in build_rooms():
    hotel.add_room(r)

print(
    f"โรงแรม '{hotel.name}' "
    f"จำนวน {len(hotel.rooms)} ห้อง"
)

stats = run_simulation(hotel)

print(f"\nคำขอจองทั้งหมด  : {stats['requests']:,} ราย")
print(f"จองสำเร็จ        : {stats['bookings']:,} ใบ")
print(f"ถูกปฏิเสธ        : {stats['rejected']:,} ราย (ห้องเต็ม)")
print(f"ลูกค้าในระบบ     : {stats['guests']:,} คน")
print(f"สถานะใบจอง      : {stats['status_counts']}")
print(f"\nOccupancy : {stats['occupancy']:.1f} %")
print(f"ADR       : {stats['adr']:,.0f} บาท")
print(f"RevPAR    : {stats['revpar']:,.0f} บาท")


In [ ]:
# ตัวอย่างใบจองและรายการที่ถูกปฏิเสธ
for b in hotel.bookings[:5]:
    print(b)
print("\nตัวอย่างรายการที่จองไว้ไม่ได้:")
for r in hotel.rejected_log[:3]:
    print(" ", r)
